# Phase III: Model Development and Evaluation
**Minor Project: Predictive Classification using Random Forest Ensembles**

Input: the four CSVs from Phase II. Steps: (1) justify `n_estimators` and `max_depth`, (2) fit the final `RandomForestClassifier`, (3) evaluate with the five required metrics, (4) compare with Gradient Boosting.

In [ ]:
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, time
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (confusion_matrix, precision_score, recall_score, f1_score, accuracy_score,
                             roc_auc_score, roc_curve, precision_recall_curve, average_precision_score,
                             classification_report)
sns.set_theme(style='whitegrid')
os.makedirs('model_figures', exist_ok=True)
RANDOM_STATE = 42

X_train = pd.read_csv('X_train.csv'); X_test = pd.read_csv('X_test.csv')
y_train = pd.read_csv('y_train.csv').squeeze(); y_test = pd.read_csv('y_test.csv').squeeze()
print('Train', X_train.shape, '| Test', X_test.shape)
print(f'Positive rate  train {y_train.mean()*100:.2f}%  test {y_test.mean()*100:.2f}%')

## 1. Hyperparameter justification
Both parameters are studied with **3-fold stratified cross-validation on the training data only** (the test set stays untouched). The score is **F1 of the minority class**, because accuracy is misleading under heavy imbalance. A stratified subsample keeps the run-time reasonable.

In [ ]:
tune_n = min(60000, len(X_train))
if tune_n < len(X_train):
    from sklearn.model_selection import train_test_split
    Xt, _, yt, _ = train_test_split(X_train, y_train, train_size=tune_n, stratify=y_train, random_state=RANDOM_STATE)
else:
    Xt, yt = X_train, y_train
cv = StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE)

def cv_f1(**p):
    rf = RandomForestClassifier(class_weight='balanced_subsample', n_jobs=-1, random_state=RANDOM_STATE, **p)
    s = cross_val_score(rf, Xt, yt, cv=cv, scoring='f1')
    return s.mean(), s.std()

n_grid = [10, 25, 50, 100, 200, 300]
d_grid = [5, 10, 15, 20, 30, None]
t0 = time.time()
n_res = [cv_f1(n_estimators=n, max_depth=None) for n in n_grid]
d_res = [cv_f1(n_estimators=100, max_depth=d) for d in d_grid]
print(f'Tuning took {time.time()-t0:.0f}s')

tbl_n = pd.DataFrame(n_res, index=n_grid, columns=['CV F1 mean','std']).rename_axis('n_estimators').round(4)
tbl_d = pd.DataFrame(d_res, index=[str(d) for d in d_grid], columns=['CV F1 mean','std']).rename_axis('max_depth').round(4)
display(tbl_n); display(tbl_d)

In [ ]:
# Parsimony rule: choose the smallest setting whose CV-F1 is within 0.01 of the best
def pick(grid, res):
    best = max(r[0] for r in res)
    for g, r in zip(grid, res):
        if r[0] >= best - 0.01: return g
best_n, best_d = pick(n_grid, n_res), pick(d_grid, d_res)

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
m, s = zip(*n_res)
ax[0].errorbar(n_grid, m, yerr=s, marker='o', capsize=3); ax[0].axvline(best_n, ls='--', c='r')
ax[0].set_xlabel('n_estimators'); ax[0].set_ylabel('CV F1 (minority class)'); ax[0].set_title('Number of trees')
m, s = zip(*d_res)
ax[1].errorbar(range(len(d_grid)), m, yerr=s, marker='o', capsize=3, color='g')
ax[1].axvline(d_grid.index(best_d), ls='--', c='r')
ax[1].set_xticks(range(len(d_grid))); ax[1].set_xticklabels([str(d) for d in d_grid])
ax[1].set_xlabel('max_depth'); ax[1].set_title('Tree depth')
plt.tight_layout(); plt.savefig('model_figures/07_hyperparameter_tuning.png', dpi=300); plt.show()

print(f'CHOSEN: n_estimators = {best_n}, max_depth = {best_d}')
print(f'''
Justification
- n_estimators={best_n}: CV-F1 rises quickly then plateaus. More trees never cause over-fitting in a Random Forest,
  but past the plateau they only cost time and memory.
- max_depth={best_d}: very shallow trees under-fit (cannot isolate a rare class that depends on elevation x soil x water
  interactions); beyond this depth the gain is within noise, so a simpler model is preferred.
- class_weight='balanced_subsample': each tree re-weights classes inside its bootstrap sample so the {y_train.mean()*100:.2f}% minority is not ignored.''')

## 2. Fit the final model on the full training set

In [ ]:
t0 = time.time()
rf = RandomForestClassifier(n_estimators=best_n, max_depth=best_d, class_weight='balanced_subsample',
                            n_jobs=-1, random_state=RANDOM_STATE).fit(X_train, y_train)
print(f'Trained in {time.time()-t0:.0f}s')
y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

## 3. Evaluation: the five required metrics

### 3.1 Confusion matrix (heatmap)

In [ ]:
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
lab = np.array([[f'TN\n{tn:,}', f'FP\n{fp:,}'], [f'FN\n{fn:,}', f'TP\n{tp:,}']])
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
sns.heatmap(cm, annot=lab, fmt='', cmap='Blues', cbar=False, ax=ax[0],
            xticklabels=['Pred: Other','Pred: Cottonwood'], yticklabels=['Actual: Other','Actual: Cottonwood'])
ax[0].set_title('Confusion matrix (counts)')
sns.heatmap(cm / cm.sum(axis=1, keepdims=True), annot=True, fmt='.2%', cmap='Greens', cbar=False, ax=ax[1],
            xticklabels=['Pred: Other','Pred: Cottonwood'], yticklabels=['Actual: Other','Actual: Cottonwood'])
ax[1].set_title('Row-normalised (per actual class)')
plt.tight_layout(); plt.savefig('model_figures/08_confusion_matrix.png', dpi=300); plt.show()

acc = accuracy_score(y_test, y_pred); baseline = 1 - y_test.mean()
print(f'TP={tp:,}  TN={tn:,}  FP={fp:,}  FN={fn:,}')
print(f'Accuracy {acc*100:.2f}% vs "always predict Other" baseline {baseline*100:.2f}% -> accuracy alone is not informative.')
print(f'{fn:,} real Cottonwood/Willow cells were MISSED; {fp:,} other cells were wrongly flagged.')

### 3.2 Precision and recall trade-off

In [ ]:
prec, rec = precision_score(y_test, y_pred, zero_division=0), recall_score(y_test, y_pred)
print(f'Precision = {prec:.3f}: when the model says Cottonwood/Willow, it is right {prec*100:.1f}% of the time (exactness).')
print(f'Recall    = {rec:.3f}: the model finds {rec*100:.1f}% of all real Cottonwood/Willow cells (completeness).')
if rec < prec:
    print(f'-> Recall < precision: the model is cautious. It misses {(1-rec)*100:.1f}% of real cells (possible unmapped riparian habitat).')
elif rec > prec:
    print(f'-> Precision < recall: the model is generous. {(1-prec)*100:.1f}% of alerts are false alarms (wasted survey effort).')

# Precision-recall curve + threshold sweep to show the trade-off explicitly
pc, rc, thr = precision_recall_curve(y_test, y_prob)
ap = average_precision_score(y_test, y_prob)
plt.figure(figsize=(6.5, 6))
plt.plot(rc, pc, lw=2, label=f'AP = {ap:.4f}')
plt.scatter([rec], [prec], c='r', zorder=5, label='Operating point (threshold 0.5)')
plt.axhline(y_test.mean(), ls='--', c='k', label='No-skill baseline')
plt.xlabel('Recall'); plt.ylabel('Precision'); plt.title('Precision-Recall curve'); plt.legend(loc='lower left')
plt.tight_layout(); plt.savefig('model_figures/09_precision_recall_curve.png', dpi=300); plt.show()

sweep = pd.DataFrame([(t, precision_score(y_test, (y_prob>=t).astype(int), zero_division=0),
                          recall_score(y_test, (y_prob>=t).astype(int)),
                          f1_score(y_test, (y_prob>=t).astype(int))) for t in [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8]],
                     columns=['threshold','precision','recall','F1']).round(3)
display(sweep)
print('Lower threshold -> higher recall, lower precision; higher threshold -> the reverse.')

### 3.3 F1-score

In [ ]:
f1 = f1_score(y_test, y_pred)
print(f'F1 = 2PR/(P+R) = {f1:.3f}')
print('The harmonic mean is dominated by the smaller of P and R, so a high F1 needs BOTH to be high,')
print('which is why it is preferred over accuracy on this imbalanced dataset.\n')
print(classification_report(y_test, y_pred, target_names=['Other','Cottonwood/Willow'], digits=3))

### 3.4 ROC curve and AUC

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)
plt.figure(figsize=(6.5, 6))
plt.plot(fpr, tpr, lw=2, label=f'Random Forest (AUC = {auc:.4f})')
plt.plot([0, 1], [0, 1], 'k--', label='Chance (AUC = 0.5)')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate'); plt.title('ROC curve'); plt.legend(loc='lower right')
plt.tight_layout(); plt.savefig('model_figures/10_roc_curve.png', dpi=300); plt.show()
print(f'ROC-AUC = {auc:.4f}: probability a random positive cell is ranked above a random negative one.')
print(f'Caveat: with so many negatives, FPR stays small and AUC looks optimistic; average precision ({ap:.3f}) is the stricter view.')

### 3.5 Feature importance

In [ ]:
imp = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)
top = imp.head(15)[::-1]
plt.figure(figsize=(9, 6))
plt.barh(top.index, top.values, color='#4C72B0')
plt.xlabel('Mean decrease in impurity'); plt.title('Top 15 feature importances')
plt.tight_layout(); plt.savefig('model_figures/11_feature_importance.png', dpi=300); plt.show()

soil_share = imp[[c for c in imp.index if c.startswith('Soil_Type')]].sum()*100
wild_share = imp[[c for c in imp.index if c.startswith('Wilderness')]].sum()*100
print('Top 3:', ', '.join(imp.head(3).index), f'-> {imp.head(3).sum()*100:.1f}% of total importance')
print(f'All 40 soil columns together: {soil_share:.1f}%   |   wilderness columns together: {wild_share:.1f}%')
print('Caveat: impurity importance favours continuous features and splits credit between correlated ones.')

## 4. Comparison with Gradient Boosting (for the conclusion)

In [ ]:
hgb = HistGradientBoostingClassifier(class_weight='balanced', random_state=RANDOM_STATE).fit(X_train, y_train)
hp, hpr = hgb.predict(X_test), hgb.predict_proba(X_test)[:, 1]
comp = pd.DataFrame({
    'Random Forest': [precision_score(y_test, y_pred, zero_division=0), recall_score(y_test, y_pred), f1_score(y_test, y_pred), roc_auc_score(y_test, y_prob)],
    'Gradient Boosting': [precision_score(y_test, hp, zero_division=0), recall_score(y_test, hp), f1_score(y_test, hp), roc_auc_score(y_test, hpr)]},
    index=['Precision','Recall','F1','ROC-AUC']).round(4)
comp

In [ ]:
# Save all numbers for the technical report
final = pd.DataFrame({'Metric':['Accuracy','Precision','Recall','F1','ROC-AUC','Average precision','TP','TN','FP','FN','n_estimators','max_depth'],
                      'Value':[acc, prec, rec, f1, auc, ap, tp, tn, fp, fn, best_n, str(best_d)]})
final.to_csv('final_results.csv', index=False); display(final)

## Phase III summary
- **Model:** `RandomForestClassifier(n_estimators=best_n, max_depth=best_d, class_weight='balanced_subsample')`, parameters chosen by cross-validated minority-class F1 on the training data only.
- **All five required metrics** are computed, plotted at 300 dpi and interpreted: confusion-matrix heatmap, precision/recall trade-off (with PR curve and threshold sweep), F1, ROC-AUC curve, feature-importance bar chart.
- **Next:** write the technical report (workflow, cleaning steps, findings, conclusion on strengths, weaknesses and improvements) using `final_results.csv` and the figures.